# Jageocoder住所→座標変換 PoC（実験用Notebook）

このNotebookは、**Jageocoderによる住所→座標変換が現在実用可能かどうかを確認するための実験用Notebook**です。

**sheltermatch本体（`sheltermatch.ipynb`）ではありません。** ここでの結果をsheltermatch本体へ統合する処理は、このNotebookには含まれていません。

## 目的

以前、sheltermatchでの住所→座標変換の検討では、以下の問題が確認されていました。

- Google Colab側のPythonバージョンとの互換性
- Jageocoder / jageocoder-converter間のAPI互換性
- 辞書生成時の外部データ取得失敗
- 配布元でのHTTP 403等

Jageocoderは継続的に更新されているため、このNotebookでは現時点で以下をゼロベースで再確認します。

1. Colabで現在のJageocoderを正常にインストール・importできるか
2. 沖縄県だけの辞書を生成できるか
3. その辞書をJageocoderから利用できるか
4. 糸満市の公開住所を住所→緯度経度へ変換できるか
5. 一致した住所と一致レベルを確認できるか

## 検証する2つの経路

PyPI公開版とjageocoder-converterの公式GitHub最新版とでは、対応するJageocoder本体のバージョンが異なります。
どちらか一方だけを試して「成立しない」と判断しないよう、このNotebookでは以下の2経路を**それぞれ隔離した
環境（`pip install --target` + サブプロセス）で独立に実行し、同じNotebook内で結果を比較**します。

- **経路A（PyPI公開版）**: `jageocoder==2.1.0` + `jageocoder-converter==2.0.3`
- **経路B（jageocoder-converter公式GitHub最新版）**: `jageocoder==2.2.1.1` + `jageocoder-converter`
  （GitHub `t-sagara/jageocoder-converter` の `v2.1.1` タグ。PyPI未公開だが`pyproject.toml`上は
  `jageocoder>=2.1.0`に対応しており、2.2系でのみ有効な内部API変更にも対応済み）

2つの経路は同一プロセス内でimportすると互いのバージョンを汚染してしまうため、それぞれ別ディレクトリへ
`pip install --target`し、`PYTHONPATH`を切り替えたサブプロセスとして実行することで分離しています。

## 進め方

上から順にすべてのセルを実行してください（「ランタイム → すべてのセルを実行」）。

外部データの取得に失敗した場合、このNotebookは失敗を握りつぶさず、経路ごとに失敗した段階・取得先・
エラー概要を表示して停止します。**別のジオコーダーへの自動切替や、非公式ミラーの利用は行いません。**
失敗そのものも、今回の検証結果として意味があります。**ただし、2経路のうち一方だけが失敗しても、
もう一方の結果を確認するまでは「Jageocoder方式は成立しない」と判断しないでください。**

## このNotebookが行わないこと

- sheltermatch本体への統合
- ABR Geocoder等、他のジオコーダーへの自動フォールバック
- Google Maps等の外部Geocoding APIの利用
- 全国辞書の作成
- 個人の住所を使ったテスト


In [ ]:
# ===== 設定 =====

# 辞書を作成する対象の都道府県コード（JIS X 0401、2桁）。沖縄県 = "47"。
PREF_CODE = "47"

# 変換テストの対象とする市区町村名（表示・確認用。辞書作成自体は都道府県単位で行う）。
TARGET_CITY = "糸満市"

# 生成した辞書・隔離インストール先の保存先プレフィックス（Colabランタイム上の一時ディレクトリ。
# ランタイムがリセットされると消える）。経路ごとに別ディレクトリを使うため、実際のパスは
# f"{DICTIONARY_DIR_PREFIX}_a" のように経路名を付けて使う。
DICTIONARY_DIR_PREFIX = "./jageocoder_okinawa_dict"
PACKAGE_DIR_PREFIX = "./jageocoder_poc_pkgs"

# 検証対象の2経路。どちらか一方だけを見て成否を判断しないよう、最低限この2つを比較する
# （他の組合せを試したい場合はここへ追加してよいが、A・Bは削除しないこと）。
ROUTES = {
    "A": {
        "label": "PyPI公開版 (jageocoder==2.1.0 + jageocoder-converter==2.0.3)",
        "pip_args": ["jageocoder==2.1.0", "jageocoder-converter==2.0.3", "sqlalchemy"],
    },
    "B": {
        "label": (
            "jageocoder-converter公式GitHub最新版 "
            "(jageocoder==2.2.1.1 + jageocoder-converter v2.1.1)"
        ),
        "pip_args": [
            "jageocoder==2.2.1.1",
            "git+https://github.com/t-sagara/jageocoder-converter.git@v2.1.1",
        ],
    },
}

# 個人情報は使用しない。糸満市内の公開されている公共施設の住所を使用する
# （リポジトリ内の要支援者テストデータ test/residents_sample_enriched.csv は架空住所のため、
# ここでは使用しない）。実際の住所と異なる場合は書き換えて構わない。
TEST_ADDRESSES = [
    "沖縄県糸満市潮崎町1丁目1番地",  # 糸満市役所
    "沖縄県糸満市字糸満673",  # 糸満市立糸満小学校
    "沖縄県糸満市真栄里1448番地",  # 糸満市立中央図書館
]

print("設定を読み込みました。")
print(f"  PREF_CODE   = '{PREF_CODE}'")
print(f"  TARGET_CITY = '{TARGET_CITY}'")
for route_name, route_info in ROUTES.items():
    print(f"  経路{route_name}: {route_info['label']}")


In [ ]:
# ===== 環境確認 =====
# Notebookを実行しているColab側のPython環境を確認する（パッケージのインストールは、
# 経路ごとに隔離した環境で行うため次のセルで行う）。

import io
import json
import os
import platform
import re
import subprocess
import sys

import pandas as pd
from google.colab import files

print(f"Python version : {sys.version}")
print(f"platform       : {platform.platform()}")

poc_status = {"python_env": True}


In [ ]:
# ===== 経路A・経路Bをそれぞれ隔離環境で実行（辞書生成→住所検索） =====
# 各経路を pip install --target=<専用ディレクトリ> で他経路と混ざらないよう別々に導入し、
# PYTHONPATHをその専用ディレクトリへ切り替えたサブプロセスとして実行する。
# こうすることで、同一Notebook内で「jageocoder==2.1.0」と「jageocoder==2.2.1.1」を
# 互いに汚染せずに両方とも実際に試すことができる（どちらか一方だけを見て判断しない）。
#
# サブプロセス内のワーカースクリプトは、辞書生成→Jageocoder初期化→TEST_ADDRESSESの検索まで
# 行い、結果を1行のJSONとして標準出力へ表示する（jageocoder_converterの進行状況ログは標準
# エラー出力に出るため、ここでは末尾数行だけを抜粋して表示し、大量の内部ログは流さない）。

MARKER = "===POC_RESULT_JSON==="

WORKER_SCRIPT = r"""
import json, os, sys

def _find_download_url(exc):
    tb = exc.__traceback__
    while tb is not None:
        value = tb.tb_frame.f_locals.get("url")
        if isinstance(value, str) and value.startswith("http"):
            return value
        tb = tb.tb_next
    return None

def _search_all(addresses):
    import jageocoder
    results = []
    for address in addresses:
        try:
            result = jageocoder.search(address)
            candidates = result.get("candidates", [])
        except Exception as e:
            results.append({"input_address": address, "matched_address": None,
                             "latitude": None, "longitude": None, "match_level": None,
                             "status": "error: " + type(e).__name__ + ": " + str(e)})
            continue
        if candidates:
            best = candidates[0]
            results.append({"input_address": address,
                             "matched_address": "".join(best.get("fullname", [])),
                             "latitude": best.get("y"), "longitude": best.get("x"),
                             "match_level": best.get("level"), "status": "matched"})
        else:
            results.append({"input_address": address, "matched_address": None,
                             "latitude": None, "longitude": None, "match_level": None,
                             "status": "no_match"})
    return results

MARKER = "===POC_RESULT_JSON==="
output = {"jageocoder_import_ok": False, "dictionary_ok": False, "geocode_ok": False,
          "results": [], "failure": None}

db_dir = os.environ["POC_DB_DIR"]
pref_code = os.environ["POC_PREF_CODE"]
mode = os.environ["POC_MODE"]
addresses = json.loads(os.environ["POC_ADDRESSES"])

try:
    import jageocoder
    import jageocoder_converter
    output["jageocoder_import_ok"] = True
except Exception as e:
    output["failure"] = {"stage": "Jageocoder / jageocoder-converterのimport",
                          "source": None, "error_type": type(e).__name__, "error": str(e)}
    print(MARKER + json.dumps(output, ensure_ascii=False))
    sys.exit(0)

if mode == "build_and_search":
    try:
        jageocoder_converter.convert(prefs=[pref_code], db_dir=db_dir, quiet=True)
        output["dictionary_ok"] = True
    except Exception as e:
        output["failure"] = {"stage": "沖縄県限定辞書の生成（jageocoder_converter.convert）",
                              "source": _find_download_url(e),
                              "error_type": type(e).__name__, "error": str(e)}
        print(MARKER + json.dumps(output, ensure_ascii=False))
        sys.exit(0)
else:
    output["dictionary_ok"] = True

try:
    jageocoder.init(db_dir=db_dir)
    output["results"] = _search_all(addresses)
    output["geocode_ok"] = any(r["status"] == "matched" for r in output["results"])
except Exception as e:
    output["failure"] = {"stage": "Jageocoder初期化・住所検索", "source": None,
                          "error_type": type(e).__name__, "error": str(e)}

print(MARKER + json.dumps(output, ensure_ascii=False))
"""


def run_worker(target_dir, db_dir, pref_code, mode, addresses):
    env = dict(os.environ)
    env["PYTHONPATH"] = target_dir + os.pathsep + env.get("PYTHONPATH", "")
    env["POC_DB_DIR"] = db_dir
    env["POC_PREF_CODE"] = pref_code
    env["POC_MODE"] = mode
    env["POC_ADDRESSES"] = json.dumps(addresses, ensure_ascii=False)

    proc = subprocess.run(
        [sys.executable, "-c", WORKER_SCRIPT], capture_output=True, text=True, env=env
    )

    log_tail = [l for l in proc.stderr.strip().splitlines() if l.strip()][-5:]
    if log_tail:
        print("  ログ(末尾抜粋):")
        for line in log_tail:
            print(f"    {line}")

    for line in proc.stdout.splitlines():
        if line.startswith(MARKER):
            return json.loads(line[len(MARKER):])

    return {
        "jageocoder_import_ok": False, "dictionary_ok": False, "geocode_ok": False,
        "results": [],
        "failure": {
            "stage": "サブプロセスの実行",
            "source": None,
            "error_type": "NoResult",
            "error": (proc.stdout + proc.stderr)[-2000:],
        },
    }


def print_route_report(route_name, label, data):
    print(f"[経路{route_name}] {label}")
    print(f"  Jageocoder本体: {'成功' if data['jageocoder_import_ok'] else '失敗'}")
    print(f"  沖縄県辞書生成: {'成功' if data['dictionary_ok'] else '失敗'}")
    if data["failure"]:
        error_text = data["failure"]["error"] or ""
        status_match = re.search(r"\b([1-5]\d{2})\b", error_text)
        print(f"  失敗段階     : {data['failure']['stage']}")
        print(f"  取得先       : {data['failure']['source'] or '特定できず'}")
        print(f"  HTTP status等: {status_match.group(1) if status_match else '不明(下記エラー概要を参照)'}")
        print(f"  エラー概要   : {data['failure']['error_type']}: {error_text}")
    else:
        matched = sum(1 for r in data["results"] if r["status"] == "matched")
        print(f"  住所検索: {len(TEST_ADDRESSES)}件中{matched}件を変換")


route_results = {}
route_target_dirs = {}

for route_name, route_info in ROUTES.items():
    print(f"\n===== 経路{route_name}: {route_info['label']} =====")
    target_dir = f"{PACKAGE_DIR_PREFIX}_{route_name.lower()}"
    db_dir = f"{DICTIONARY_DIR_PREFIX}_{route_name.lower()}"
    route_target_dirs[route_name] = target_dir

    print(f"  パッケージを隔離ディレクトリへ導入します: {target_dir}")
    install = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--target", target_dir]
        + route_info["pip_args"],
        capture_output=True,
        text=True,
    )
    if install.returncode != 0:
        data = {
            "jageocoder_import_ok": False, "dictionary_ok": False, "geocode_ok": False,
            "results": [],
            "failure": {
                "stage": "パッケージ導入（pip install --target）",
                "source": None,
                "error_type": "InstallError",
                "error": install.stderr[-2000:],
            },
        }
    else:
        print(
            "  quiet=Trueのため、各データ配布元の利用規約への同意プロンプトは表示されません。"
            "実行前に、国土数値情報・国土地理院・アドレス・ベース・レジストリ等、"
            "jageocoder_converterが参照する各データ配布元の利用規約を確認してください。"
        )
        data = run_worker(target_dir, db_dir, PREF_CODE, "build_and_search", TEST_ADDRESSES)

    route_results[route_name] = data
    print_route_report(route_name, route_info["label"], data)


In [ ]:
# ===== 結果確認 =====
# Jageocoderのsearch()が実際に返す値（matched/candidates. 各候補はid/name/x/y/level/priority/
# note/fullname）だけを表示する。Jageocoderが返していない精度は独自に推測しない。
# match_levelの説明は、Jageocoder自身のAddressLevel定数（1~8の意味はjageocoder.address.
# AddressLevelのdocstringに定義されている。経路A・Bで値は共通）をそのまま使う。

# AddressLevelのdocstringに定義された1~8の意味そのもの（独自の精度解釈は加えない）。
LEVEL_LABELS = {
    1: "都道府県",
    2: "郡・支庁・振興局",
    3: "市町村および特別区",
    4: "政令市の区",
    5: "大字",
    6: "字",
    7: "地番または住居表示実施地域の街区",
    8: "枝番または住居表示実施地域の住居番号",
}

comparison_rows = []
for route_name, data in route_results.items():
    for row in data["results"]:
        comparison_rows.append({"route": route_name, **row})

if not comparison_rows:
    print("いずれの経路も住所検索まで到達しませんでした。上の各経路の実行結果を確認してください。")
else:
    comparison_df = pd.DataFrame(comparison_rows)
    comparison_df["match_level_label"] = comparison_df["match_level"].apply(
        lambda level: LEVEL_LABELS.get(int(level)) if pd.notna(level) else None
    )
    display(
        comparison_df[
            [
                "route",
                "input_address",
                "matched_address",
                "latitude",
                "longitude",
                "match_level",
                "match_level_label",
                "status",
            ]
        ]
    )


In [ ]:
# ===== CSVアップロードによる簡易確認（検証用） =====
# 任意のセル。id,address列を持つCSVをアップロードすると、住所検索まで成立した経路のうち
# 1つを使って一括変換を試せる（辞書生成からやり直さず、既に作成済みの辞書をそのまま使う）。
# あくまでPoCの成立確認用であり、sheltermatch本体のCSV仕様との統合はここでは行わない。
# 変換できなかった行も削除しない。アップロードをキャンセルした場合はこのセルをスキップする。

_usable_routes = [name for name, data in route_results.items() if data.get("geocode_ok")]
# 複数の経路が使える場合は、より新しい経路（B）を優先する。
_selected_route = "B" if "B" in _usable_routes else (_usable_routes[0] if _usable_routes else None)

if _selected_route is None:
    print("住所検索まで成立した経路がないため、このセルはスキップします。")
    poc_status["csv_batch"] = None
else:
    print(f"経路{_selected_route}（{ROUTES[_selected_route]['label']}）を使って一括変換します。")
    print("id,address列を持つCSVを選択してください（試さない場合はアップロードをキャンセルしてください）。")
    uploaded_csv = files.upload()

    if not uploaded_csv:
        print("CSVがアップロードされなかったため、このセルはスキップされました。")
        poc_status["csv_batch"] = None
    else:
        csv_filename = list(uploaded_csv.keys())[0]
        addresses_df = pd.read_csv(io.BytesIO(uploaded_csv[csv_filename]))

        if "address" not in addresses_df.columns:
            print("'address'列が見つかりません。id,addressの形式で用意してください。")
            poc_status["csv_batch"] = False
        else:
            target_dir = route_target_dirs[_selected_route]
            db_dir = f"{DICTIONARY_DIR_PREFIX}_{_selected_route.lower()}"
            batch_addresses = addresses_df["address"].astype(str).tolist()
            batch_result = run_worker(target_dir, db_dir, PREF_CODE, "search_only", batch_addresses)

            if batch_result["failure"]:
                print("一括変換に失敗しました。")
                print(f"エラー概要: {batch_result['failure']['error_type']}: {batch_result['failure']['error']}")
                poc_status["csv_batch"] = False
            else:
                batch_df = pd.DataFrame(batch_result["results"])
                addresses_df["matched_address"] = batch_df["matched_address"]
                addresses_df["latitude"] = batch_df["latitude"]
                addresses_df["longitude"] = batch_df["longitude"]
                addresses_df["match_level"] = batch_df["match_level"]
                addresses_df["geocode_status"] = batch_df["status"]

                matched_count = int((addresses_df["geocode_status"] == "matched").sum())
                print(f"{len(addresses_df)}行中{matched_count}行を変換しました。")
                display(addresses_df.head())
                poc_status["csv_batch"] = matched_count > 0


In [ ]:
# ===== 検証結果サマリ =====


def _fmt(value):
    if value is True:
        return "OK"
    if value is False:
        return "NG"
    return "未実施"


print("Jageocoder方式PoC 検証結果サマリ")
print(f"  Python環境 : {_fmt(poc_status.get('python_env'))}")

any_dictionary_ok = False
any_geocode_ok = False
for route_name, route_info in ROUTES.items():
    data = route_results.get(route_name, {})
    print(f"\n  [経路{route_name}] {route_info['label']}")
    print(f"    Jageocoder import : {_fmt(data.get('jageocoder_import_ok'))}")
    print(f"    沖縄県限定辞書    : {_fmt(data.get('dictionary_ok'))}")
    print(f"    住所検索          : {_fmt(data.get('geocode_ok'))}")
    any_dictionary_ok = any_dictionary_ok or bool(data.get("dictionary_ok"))
    any_geocode_ok = any_geocode_ok or bool(data.get("geocode_ok"))

print(f"\n  CSV一括変換(選択経路: {_selected_route or '未実施'}) : {_fmt(poc_status.get('csv_batch'))}")
print()

if any_geocode_ok:
    print("いずれかの経路で辞書生成・住所検索まで成立しています。Jageocoder方式を次の検証へ進められる材料があります。")
elif any_dictionary_ok:
    print("辞書生成までは成立した経路がありますが、住所検索までは成立しませんでした。各経路の出力を確認してください。")
else:
    print("経路A・経路Bのいずれも辞書生成段階で成立しませんでした。上の各経路の出力（失敗段階・取得先・エラー概要）を確認してください。")
    print("2経路とも失敗したことを確認したうえでの結論であり、片方の経路だけを見た判断ではありません。")

print()
print("この結果は本Notebook内の検証にとどまり、sheltermatch本体へは反映していません。")
